# Learning Objectives

* Build a multi-source environmental monitoring workflow from Python wrapper scripts and an Apptainer container.
* Generate the workflow's catalogs and DAG with the same generator class the command-line tool uses.
* Visualize a workflow DAG before submitting it.
* Plan, submit, monitor, and pull statistics for a Pegasus workflow from a notebook.
* Inspect the decision layers and the dashboard produced by a run.

# Drought / Snowmelt / Wildfire Monitoring Workflow

This workflow turns field observations from up to three sources — SAGE (Waggle edge sensors), GLEES (the
AmeriFlux US-GLE flux tower) and SNOTEL (NRCS snow telemetry) — into five decision layers for a mountain watershed,
then renders them into one dashboard. Each source has its own fetch job; `harmonize` merges whatever they return, so
the workflow still runs when a source is unavailable. See `README.md` for the full description and data sources.

**Workflow Jobs:**
1. `fetch_sage_data`, `fetch_glees_data`, `fetch_snotel_data` - fetch each selected source (with retries)
2. `harmonize` - merge the sources onto one time base
3. `soil_moisture_map`, `forest_water_stress`, `snowmelt_recharge` - first-order layers
4. `drought_warning`, `wildfire_risk` - layers built from the ones above
5. `visualize_dashboard` - one PNG over all five layers

**Prerequisites:** [Pegasus WMS](https://pegasus.isi.edu/) >= 5.0, [HTCondor](https://htcondor.org/) >= 10.2,
Python 3.8+ with `pyyaml`, and [Apptainer](https://apptainer.org/) — all assumed to already be available in
this notebook's environment.

## 1. Choose the Data Sources

The data is fetched at run time by the workflow's own jobs, so there is nothing to download first. This notebook
uses **SNOTEL only**: it is fully public and needs no credentials, which makes it the README's recommended first
test. GLEES needs AmeriFlux credentials (`AMERIFLUX_USER_ID`, `AMERIFLUX_USER_EMAIL`, set in the environment before generating) or a
pre-downloaded BASE CSV; add `"glees"` and `"sage"` to `SOURCES` once you have them.

In [ ]:
SOURCES = ["snotel"]   # any of "sage", "glees", "snotel"
CONFIG = "region_config.json"

## 2. Build the Apptainer Container

The workflow's jobs run inside one Apptainer container (`Apptainer/Drought_Container.def`) with Python, pandas,
NumPy and matplotlib. Building it takes a few minutes; skip the rebuild if the `.sif` already exists.

In [ ]:
%%bash
if [ ! -f Apptainer/Drought_Container.sif ]; then
    apptainer build Apptainer/Drought_Container.sif Apptainer/Drought_Container.def
else
    echo "Container already built: Apptainer/Drought_Container.sif"
fi

## 3. Generate the Workflow

The following cell uses the `DroughtWorkflow` class from `workflow_generator.py` — the same generator used by the
command-line tool — to build the Pegasus workflow DAG (site, transformation, and replica catalogs) and writes it to
`workflow.yml`. The workflow-specific inputs are the same values the CLI's `--config`/`--sources` options set.

`create_sites_catalog()` writes a self-contained site catalog for this machine's HTCondor pool (`local` +
`compute`). To run on a resource with a centrally hosted site catalog
([pegasus-site-catalogs](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf), e.g.
`access-pegasus.yml`, `unity.yml`), set `HOSTED_SITE_CATALOG` instead — or set it once in `~/.pegasusrc`.

In [ ]:
import argparse
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))

from workflow_generator import DroughtWorkflow

DAGFILE = "workflow.yml"
EXEC_SITE = "compute"
HOSTED_SITE_CATALOG = None  # e.g. "unity.yml"; None = local HTCondor catalog below

# The inputs the CLI would parse (start/end dates default to the config's range)
ARGS = argparse.Namespace(config=CONFIG, sources=SOURCES, start_date=None,
                          end_date=None, glees_base_csv=None)

workflow = DroughtWorkflow(dagfile=DAGFILE)

print("Creating workflow properties...")
workflow.create_pegasus_properties(hosted_site_catalog=HOSTED_SITE_CATALOG)

if HOSTED_SITE_CATALOG is None:
    print("Creating execution sites...")
    workflow.create_sites_catalog(exec_site_name=EXEC_SITE)

print("Creating transformation catalog...")
workflow.create_transformation_catalog(
    container_sif="Apptainer/Drought_Container.sif", exec_site_name=EXEC_SITE)

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating drought workflow DAG...")
workflow.create_workflow(ARGS)

workflow.write()
print("\nDrought Workflow has been generated!")

## 4. View the Generated Workflow DAG

Before submitting, visualize the workflow DAG using `pegasus-graphviz`. Each selected fetch job feeds `harmonize`,
which fans out to the three first-order layers; those feed the two derived layers and the dashboard.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image

Image(filename="workflow.png")

## 5. Plan and Submit the Workflow

The workflow will be planned and submitted for execution on the `compute` site defined in the site catalog chosen
above.

The CLI version of this generator (`workflow_generator.py`) writes the same workflow non-interactively and prints
the `pegasus-plan` command; it never submits by itself. Submitting is this explicit step.

In [ ]:
workflow.plan_submit(exec_site_name=EXEC_SITE)

## Workflow Status Monitoring

After successful submission, monitor the workflow status. The output shows job counts and idle/running/completed
states. `wait()` blocks (with a progress bar) until the workflow completes or fails.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 6. Statistics

Once the workflow completes, pull execution statistics from the Pegasus provenance database.

In [ ]:
workflow.statistics()

## 7. Examine the Results

The workflow stages the following final outputs to `output/` (to `wf-output/` instead when
`HOSTED_SITE_CATALOG` is set: the hosted catalog has no `local` site, so Pegasus uses its default one):

| Output | Description |
|--------|-------------|
| `snotel_observations.csv` (and `sage_`/`glees_observations.csv` when selected) | Raw observations per source |
| `observations.csv`, `harmonization_report.json` | Harmonized observations and what each source contributed |
| `soil_moisture_map.json`, `forest_water_stress.json`, `snowmelt_recharge.json` | First-order layers |
| `drought_warning.json`, `wildfire_risk.json` | Derived layers |
| `drought_dashboard.png` | Dashboard over all five layers |

In [ ]:
!ls -lt output/

The dashboard over all five layers (a SNOTEL-only run has no flux or soil-moisture data, so those layers show their
reduced-input behavior):

In [ ]:
Image(filename="output/drought_dashboard.png")